# Item-to-Item Product Recommendation by Collaborative Filtering

Data source: Online Retail Data Set from UCI Machine Learning Repository (http://archive.ics.uci.edu/ml/datasets/online+retail)

**Load & Prepare the Data**

In [ ]:
# Import necessary libraries.
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Read data source Excel files.
df1 = pd.read_excel('./data/Online Retail.xlsx')

In [ ]:
# Check dataframe information.
df1.info()

In [ ]:
# Read header of dataframe.
df1.head()

In [ ]:
# Check any column containing the null value.
df1.isnull().any()

In [ ]:
# Count the number of null value records in the CustomerID column.
df1['CustomerID'].isna().sum()

In [ ]:
df1a = df1.dropna(subset=['CustomerID'])

In [ ]:
# Check dataframe information.
df1a.info()

In [ ]:
# Read header of dataframe.
df1a.head()

In [ ]:
# Create CustomerID vs Item (Purchased Items, by StockCode) matrix by pivot table function.
CustomerID_Item_matrix = df1a.pivot_table(
    index='CustomerID',
    columns='StockCode',
    values='Quantity',
    aggfunc='sum'
)

In [ ]:
# Display the shape of matrix, 4372 rows of CustomerID, 3684 columns of Item.
CustomerID_Item_matrix.shape

In [ ]:
# Update illustration of the matrix, 1 to represent customer have purchased item, 0 to represent customer haven't purchased.
CustomerID_Item_matrix = CustomerID_Item_matrix.applymap(lambda x: 1 if x > 0 else 0)

In [ ]:
# Read header of CustomerID vs Item matrix.
CustomerID_Item_matrix.loc[12680:].head()

**Calculate the Item to Item similarity by Cosine Similarity**

In [ ]:
# Create Item to Item similarity matrix.
item_item_similarity_matrix = pd.DataFrame(
    cosine_similarity(CustomerID_Item_matrix.T)
)

In [ ]:
# Display header of Item to Item similarity matrix.
item_item_similarity_matrix.head()

In [ ]:
# Update index to corresponding Item Code (StockCode).
item_item_similarity_matrix.columns = CustomerID_Item_matrix.T.index
item_item_similarity_matrix['StockCode'] = CustomerID_Item_matrix.T.index
item_item_similarity_matrix = item_item_similarity_matrix.set_index('StockCode')

In [ ]:
# Display header of Item to Item similarity matrix.
item_item_similarity_matrix.head()

In [ ]:
# Randomly pick StockCode (23166) to display the most similar StockCode.
top_10_similar_items = list(
    item_item_similarity_matrix\
        .loc[23166]\
        .sort_values(ascending=False)\
        .iloc[:10]\
    .index
)

In [ ]:
# Display top 10 similar items of StockCode (23166).
top_10_similar_items

In [ ]:
# Display the list of similar items of StockCode (23166) with item Description.
df1a.loc[
    df1a['StockCode'].isin(top_10_similar_items),
    ['StockCode', 'Description']
].drop_duplicates().set_index('StockCode').loc[top_10_similar_items]

In [ ]:
# And optionally, use the heatmap to display the Item to Item similarity matrix.
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize = (30,30))
ax = sns.heatmap(
    item_item_similarity_matrix,
    vmin=-1, vmax=1, center=0,
    square=True)